# imports

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score
import mlflow
import optuna
import matplotlib.pyplot as plt

c:\Users\Manindra\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# config

In [2]:
ASSET = "BTC"
INTERVAL = "1h"

# mlflow

In [ ]:
mlflow.set_tracking_uri("http://localhost:5000")

experiment_name = f"{ASSET}_{INTERVAL}_Tuning"
mlflow.set_experiment(experiment_name)

2026/05/09 12:36:34 INFO mlflow.tracking.fluent: Experiment with name 'BTC_1h_Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='/mlruns/10', creation_time=1778326594058, experiment_id='10', last_update_time=1778326594058, lifecycle_stage='active', name='BTC_1h_Tuning', tags={}, trace_location=None, workspace='default'>

# load data

In [3]:
train_df = pd.read_parquet('../../../data/processed/train_btc_1h.parquet')
test_df = pd.read_parquet('../../../data/processed/test_btc_1h.parquet')

# splitting

In [4]:
features = [col for col in train_df.columns if col not in ['target_1h', 'target_direction']]
X_train = train_df[features]
y_train = train_df['target_direction']
X_test = test_df[features]
y_test = test_df['target_direction']

# func for tuning with mlflow,xgboost

In [7]:
def find_best_params(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'eval_metric': 'logloss',
        'random_state': 42
    }
    
    model = xgb.XGBClassifier(**param)
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    accuracy = accuracy_score(y_test, preds)
    
    with mlflow.start_run(nested=True):
        mlflow.log_params(param)
        mlflow.log_metric("accuracy", accuracy)
        
    return accuracy

print("doone")    

doone


# hyperparameter tuning with optuna

In [8]:
print(" starting hyperparameter tuning with 50 trials...")

study = optuna.create_study(direction='maximize', study_name=f"XGB_Tuning_{ASSET}")

study.optimize(find_best_params, n_trials=50)

print(f"\n Tuning completed Best Accuracy found: {study.best_value * 100:.2f}%")
print(f"Winning Parameters: {study.best_params}")

[I 2026-05-09 12:47:07,702] A new study created in memory with name: XGB_Tuning_BTC


 starting hyperparameter tuning with 50 trials...


[I 2026-05-09 12:47:10,475] Trial 0 finished with value: 0.5203480213303396 and parameters: {'n_estimators': 237, 'max_depth': 7, 'learning_rate': 0.012579771510881169, 'subsample': 0.6381923870887256}. Best is trial 0 with value: 0.5203480213303396.


🏃 View run unique-snipe-821 at: http://localhost:5000/#/experiments/10/runs/a4197c0e0a5b46528f9567e034373b17
🧪 View experiment at: http://localhost:5000/#/experiments/10


[I 2026-05-09 12:47:11,394] Trial 1 finished with value: 0.5247450650201141 and parameters: {'n_estimators': 74, 'max_depth': 6, 'learning_rate': 0.03554650569815949, 'subsample': 0.6454733603139171}. Best is trial 1 with value: 0.5247450650201141.


...

🏃 View run burly-rat-482 at: http://localhost:5000/#/experiments/10/runs/6042fe5018864ded96f3109a274f81e6
🧪 View experiment at: http://localhost:5000/#/experiments/10


[I 2026-05-09 12:48:09,359] Trial 48 finished with value: 0.5260548227149406 and parameters: {'n_estimators': 152, 'max_depth': 3, 'learning_rate': 0.050888702304717835, 'subsample': 0.937880422852245}. Best is trial 9 with value: 0.5310131911310694.


🏃 View run adaptable-panda-360 at: http://localhost:5000/#/experiments/10/runs/186b6e92527d43af8fcb0d36d9801568
🧪 View experiment at: http://localhost:5000/#/experiments/10


[I 2026-05-09 12:48:10,535] Trial 49 finished with value: 0.5221255496304612 and parameters: {'n_estimators': 195, 'max_depth': 5, 'learning_rate': 0.06522107541185021, 'subsample': 0.8647395487689408}. Best is trial 9 with value: 0.5310131911310694.


🏃 View run capricious-shoat-569 at: http://localhost:5000/#/experiments/10/runs/41eaadaded3f4eec8644960815153409
🧪 View experiment at: http://localhost:5000/#/experiments/10

 Tuning completed Best Accuracy found: 53.10%
Winning Parameters: {'n_estimators': 172, 'max_depth': 4, 'learning_rate': 0.027155572856517853, 'subsample': 0.5249636363159839}
